###### Credit
Fork from 

: https://www.kaggle.com/code/rsuhara/ai-generated-text-detection-quick-baseline

Inspired by : https://www.kaggle.com/code/yekenot/llm-detect-by-regression

rm LR and more data Ref. : https://www.kaggle.com/code/xiaocao123/ai-generated-text-detection-add-new-data

use 3 sgd Ref. : https://www.kaggle.com/code/chenbaoying/ai-generated-text-detection-3sgd-0-903

fit the tf-idf only on test data Ref. : https://www.kaggle.com/code/nahman/0-908-don-t-try-this-at-home

**[0.911]** Only using test set to extract feature from TfidfVectorizer, Inspired by the discussion in: https://www.kaggle.com/competitions/llm-detect-ai-generated-text/discussion/455701, My discussion: https://www.kaggle.com/competitions/llm-detect-ai-generated-text/discussion/455997

weights from : https://www.kaggle.com/code/siddhvr/llm-detect-ai-gt-sub

**[0.918]** Remove typos in test.csv, reference: https://www.kaggle.com/code/xiaocao123/ai-generated-text-detection-add-new-data and https://www.kaggle.com/code/murugesann/nm-llm-detect-ai-text-typo-correct-with-testdata

add this dataset :

https://www.kaggle.com/datasets/thedrcat/daigt-proper-train-dataset

https://www.kaggle.com/datasets/thedrcat/daigt-v2-train-dataset/

In [ ]:
!pip install -q language-tool-python --no-index --find-links ../input/daigt-misc/
!mkdir -p /root/.cache/language_tool_python/
!cp -r /kaggle/input/daigt-misc/lang57/LanguageTool-5.7 /root/.cache/language_tool_python/LanguageTool-5.7

# Importing library

In [ ]:
import numpy as np
import pandas as pd
import re
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, accuracy_score, roc_auc_score
import language_tool_python
from concurrent.futures import ProcessPoolExecutor

# seed = 2023
isGridSearch = False

# ALL weights caculate by rerunning grid search
# weights = [0.10526315789473684, 0.8947368421052632] 
# weights = [0.01,0.99]
# weights=[0.01,0.33,0.33,0.33]

In [ ]:
tool = language_tool_python.LanguageTool('en-US')

def correct_sentence(sentence):
    return tool.correct(sentence)

def correct_df(df):
    with ProcessPoolExecutor() as executor:
        df['text'] = list(executor.map(correct_sentence, df['text']))

# Load datasets

In [ ]:
# external_df = pd.read_csv("/kaggle/input/daigt-external-dataset/daigt_external_dataset.csv", sep=',')
train = pd.read_csv("/kaggle/input/daigt-v2-train-dataset/train_v2_drcat_02.csv")
# train_data = train[train.RDizzl3_seven == False].reset_index(drop=True)
train_data = train[train["label"]==1].sample(8000)
train_data.head(10)
train_data.count()

In [ ]:
train = train[train.RDizzl3_seven == True].reset_index(drop=True)
train = pd.concat([train,train_data])
train['text'] = train['text'].str.replace('\n', '')

test = pd.read_csv('/kaggle/input/llm-detect-ai-generated-text/test_essays.csv')
test['text'] = test['text'].str.replace('\n', '')
correct_df(test)

# addtrain1 = pd.read_csv("/kaggle/input/daigt-proper-train-dataset/train_drcat_04.csv")
train.value_counts("label")

# Preprocess and merge datasets

In [ ]:
# Combine train and test text
df = pd.concat([train['text'], test['text']], axis=0)
df.head(10)

# Feature extraction

In [ ]:
# Extract Feature
min_ngram = 3
max_ngram = 6 
vectorizer = TfidfVectorizer(ngram_range=(min_ngram, max_ngram),sublinear_tf=True)
vectorizer = vectorizer.fit(test['text'])
X = vectorizer.transform(df)

# Model initialization

In [ ]:
# lr_model = LogisticRegression()
sgd_model = SGDClassifier(max_iter=5000, tol=1e-3, loss="modified_huber") 
sgd_model2 = SGDClassifier(max_iter=5000, tol=1e-3, loss="modified_huber", class_weight="balanced")
sgd_model3 = SGDClassifier(max_iter=10000, tol=5e-4, loss="modified_huber", early_stopping=True)
# rf_model = RandomForestClassifier(n_estimators=100)
# nb_model = MultinomialNB()

# Create the ensemble model

In [ ]:
ensemble = VotingClassifier(estimators=[#('lreg', lr_model), 
                                        #('rf', rf_model),
                                        ('sgd', sgd_model), 
                                        ('sgd2', sgd_model2),
                                        ('sgd3', sgd_model3),
                                        #('nb', nb_model)
                                       ],
#                             weights=weights,
                            voting='soft')

# Define a range of weights

In [ ]:
if not isGridSearch:
    ensemble.fit(X[:train.shape[0]], train.label)
    preds_test = ensemble.predict_proba(X[train.shape[0]:])[:,1]
else:
    weights = np.linspace(0, 1, 20)
    weight_combinations = [(w, 1-w) for w in weights]

    # Define the parameter grid
    param_grid = {'weights': weight_combinations}

    # Define a scorer, for example, accuracy
    scorer = make_scorer(roc_auc_score)

    # Initialize GridSearchCV
    grid_search = GridSearchCV(estimator=ensemble, 
                               param_grid=param_grid, 
                               scoring=scorer, 
                               cv=5)

    # Fit the grid search to the data
    grid_search.fit(X[:train.shape[0]], train.label)

    # Find the best parameters
    best_weights = grid_search.best_params_['weights']
    print(f"Best Weights: {best_weights}")
    preds_test = grid_search.predict_proba(X[train.shape[0]:])[:, 1]

# Predictions

In [ ]:
pd.DataFrame({'id': test["id"], 'generated': preds_test}).to_csv('submission.csv', index=False)